# All of annplyr

## Contents

- [Set up Data](#set-up-data)
- [Filter](#filter)
- [Select](#select)
- [Arrange, Slice, Mutate, and Summarize](#arrange-slice-mutate-and-summarize)
- [Group By](#group-by)
- [Tidy Tables](#tidy-tables)
- [Pipe](#pipe)

This notebook gives a practical tour of `annplyr`. `annplyr` is inspired by [annsel](https://github.com/srivarra/annsel) and extends its predicate-based selection to the full `dplyr`/`tidyr` verb set for R tidyverse users working in Python single-cell pipelines.

`annplyr` verbs return AnnData objects when they can preserve AnnData axis
alignment. Exports such as `to_df`, `to_tidy`, `pivot_wider`, and the
pandas-only tidyr helpers return pandas data frames.

```{note}
You should be familiar with AnnData before using this notebook.
```

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata",
#     "annplyr",
#     "numpy",
#     "pandas",
# ]
# ///

## Set up Data

Importing `annplyr` registers the `ap` accessor on AnnData objects. The
example data below is intentionally small so every operation is easy to
inspect.

In [2]:
import annplyr as ap
import numpy as np
import pandas as pd
from anndata import AnnData

In [3]:
x = np.array(
    [
        [2.0, 0.0, 1.0, 4.0, 0.0],
        [0.0, 3.0, 2.0, 1.0, 5.0],
        [4.0, 1.0, 0.0, 0.0, 2.0],
        [1.0, 4.0, 3.0, 2.0, 1.0],
        [5.0, 0.0, 2.0, 3.0, 0.0],
        [0.0, 2.0, 5.0, 1.0, 4.0],
        [3.0, 1.0, 1.0, 0.0, 3.0],
        [2.0, 5.0, 0.0, 4.0, 1.0],
    ]
)

obs = pd.DataFrame(
    {
        "Cell_label": ["CD8 T", "B cell", "CD8 T", "Mono", "B cell", "Mono", "CD4 T", "CD4 T"],
        "donor_id": ["AML1", "AML1", "AML2", "AML2", "AML3", "AML3", "AML1", "AML2"],
        "sex": ["male", "female", "male", "female", "female", "male", "male", "female"],
        "disease": ["AML", "AML", "control", "AML", "control", "AML", "control", "AML"],
        "score": [0.7, 0.2, 0.9, 0.5, 0.4, 0.8, 0.6, 0.3],
        "qc_mito": [0.08, 0.12, 0.04, 0.22, 0.16, 0.07, 0.10, 0.18],
        "qc_ribo": [0.30, 0.18, 0.34, 0.28, 0.42, 0.25, 0.21, 0.38],
    },
    index=pd.Index([f"cell_{i:03d}" for i in range(8)], name="cell"),
)

var = pd.DataFrame(
    {
        "feature_name": ["CD3E", "MS4A1", "LYZ", "MALAT1", "PPBP"],
        "feature_type": ["protein_coding", "protein_coding", "protein_coding", "lncRNA", "protein_coding"],
        "vst.mean": [3.2, 2.8, 4.5, 1.4, 2.1],
        "chrom": ["chr11", "chr11", "chr12", "chr11", "chr7"],
    },
    index=pd.Index(["CD3E", "MS4A1", "LYZ", "MALAT1", "PPBP"], name="gene"),
)

adata = AnnData(X=x, obs=obs, var=var)
adata.layers["arcsinh"] = np.arcsinh(adata.X)
adata.obsm["X_pca"] = np.array(
    [
        [0.4, 1.2],
        [-0.1, 0.3],
        [0.8, -0.2],
        [1.3, 0.5],
        [0.2, -0.8],
        [-0.5, 1.1],
        [0.6, 0.0],
        [1.0, -0.4],
    ]
)
adata.varm["loadings"] = np.array(
    [
        [0.5, 1.0],
        [0.2, 0.8],
        [1.2, 0.1],
        [-0.4, 0.3],
        [0.0, 1.4],
    ]
)
adata.uns["title"] = "Small annplyr tutorial AnnData"
adata

AnnData object with n_obs × n_vars = 8 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

The accessor is available as `adata.ap`.

In [4]:
hasattr(adata, "ap")

True

## Filter

`filter` subsets observations and variables. Predicates can read from
`obs`, `var`, observation names, variable names, `X` or a layer, and keyed
`obsm`/`varm` matrices.

Here we filter by a cell label stored in `obs["Cell_label"]`.

In [5]:
adata.ap.filter(obs=ap.col("Cell_label") == "CD8 T")

View of AnnData object with n_obs × n_vars = 2 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

This is equivalent to standard AnnData indexing:

In [6]:
adata[adata.obs["Cell_label"] == "CD8 T", :]

View of AnnData object with n_obs × n_vars = 2 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

Narwhals expressions combine with `&` and `|`; a tuple of predicates applies as an implicit `AND`.

In [7]:
adata.ap.filter(
    obs=(
        ap.col("Cell_label").is_in(["CD8 T", "Mono"]),
        ap.col("sex") == "male",
    )
)

View of AnnData object with n_obs × n_vars = 3 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

In [8]:
adata.ap.filter(obs=(ap.col("Cell_label").is_in(["CD8 T", "Mono"])) | (ap.col("disease") == "control"))

View of AnnData object with n_obs × n_vars = 6 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

Filtering on `var` subsets features.

In [9]:
adata.ap.filter(var=ap.col("vst.mean") >= 3)

View of AnnData object with n_obs × n_vars = 8 × 2
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

Filtering on `X` reads gene expression by variable name. Passing `layer` redirects the expression source to that layer.

In [10]:
adata.ap.filter(
    x=ap.col("LYZ") > 1,
    layer="arcsinh",
)

View of AnnData object with n_obs × n_vars = 4 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

Sources can be combined in one call.

In [11]:
adata.ap.filter(
    obs=(ap.col("disease") == "AML", ap.col("score") >= 0.5),
    var=ap.col("feature_type") == "protein_coding",
    obsm={"X_pca": ap.col("0") > 0},
)

View of AnnData object with n_obs × n_vars = 2 × 4
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

`obs_names` and `var_names` accept dedicated expression objects that match against axis labels rather than metadata columns.

In [12]:
adata.ap.filter(
    obs_names=ap.obs_names.str.starts_with("cell_00"),
    var_names=ap.var_names.str.starts_with("C"),
)

View of AnnData object with n_obs × n_vars = 8 × 1
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

The tidyselect-aware `if_any` and `if_all` helpers reduce predicates across selected columns.

In [13]:
adata.ap.filter(obs=ap.if_any(ap.starts_with("qc_"), lambda column: ap.col(column) > 0.4)).obs[
    ["Cell_label", "qc_mito", "qc_ribo"]
]

,Cell_label,qc_mito,qc_ribo
cell,,,
cell_004,B cell,0.16,0.42


## Select

`select` keeps metadata columns and matrix features. It returns an AnnData
object whose `obs`, `var`, `X`, layers, and aligned matrices are still
shape-consistent.

In [14]:
adata.ap.select(
    obs=["Cell_label", "sex"],
    var=["feature_name", "feature_type"],
    x=["CD3E", "MS4A1"],
)

AnnData object with n_obs × n_vars = 8 × 2
    obs: 'Cell_label', 'sex'
    var: 'feature_name', 'feature_type'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

Tidyselect-style helpers make common selections compact.

In [15]:
adata.ap.select(
    obs=[ap.all_of(["Cell_label"]), ap.any_of(["missing", "score"]), ap.starts_with("qc_")],
    var=ap.ends_with("type"),
    x=ap.contains("A"),
)

AnnData object with n_obs × n_vars = 8 × 2
    obs: 'Cell_label', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_type'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

`pick` is useful in extraction contexts where you want a selector object.

In [16]:
adata.ap.to_df(obs=ap.pick(ap.starts_with("qc_"))).head()

,qc_mito,qc_ribo
cell,,
cell_000,0.08,0.30
cell_001,0.12,0.18
cell_002,0.04,0.34
cell_003,0.22,0.28
cell_004,0.16,0.42


## Arrange, Slice, Mutate, and Summarize

`arrange` reorders axes, and `slice_*` helpers take position-based subsets.

In [17]:
adata.ap.arrange(obs=ap.desc("score"), var="vst.mean")

View of AnnData object with n_obs × n_vars = 8 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

In [18]:
adata.ap.slice_head(n=3)

View of AnnData object with n_obs × n_vars = 3 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

In [19]:
adata.ap.slice_max(ap.col("score"), n=2)

View of AnnData object with n_obs × n_vars = 2 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

`mutate` writes new `obs` or `var` metadata. Matrix, layer, `obsm`, and `varm` sources are read-only inputs for new metadata columns.

In [20]:
adata.ap.mutate(
    obs={"score_percent": ap.col("score") * 100},
    x={"LYZ_counts": ap.col("LYZ")},
    obsm={"X_pca": {"pc0": ap.col("0")}},
).obs[["Cell_label", "score_percent", "LYZ_counts", "pc0"]].head()

,Cell_label,score_percent,LYZ_counts,pc0
cell,,,,
cell_000,CD8 T,70.0,1.0,0.4
cell_001,B cell,20.0,2.0,-0.1
cell_002,CD8 T,90.0,0.0,0.8
cell_003,Mono,50.0,3.0,1.3
cell_004,B cell,40.0,2.0,0.2


`across` expands one expression over selected columns.

In [21]:
adata.ap.mutate(
    obs=ap.across(ap.starts_with("qc_"), lambda column: ap.col(column) < 0.35, names="{col}_pass")
).obs.filter(like="_pass").head()

,qc_mito_pass,qc_ribo_pass
cell,,
cell_000,True,True
cell_001,True,True
cell_002,True,True
cell_003,True,True
cell_004,True,False


`summarize` reduces to a pandas DataFrame rather than returning an AnnData object.

In [22]:
adata.ap.summarize(
    obs={"cells": ap.n(), "mean_score": ap.mean("score"), "donors": ap.n_distinct("donor_id")},
    by="Cell_label",
)

,Cell_label,cells,mean_score,donors
0,CD8 T,2,0.80,2
1,B cell,2,0.30,2
2,Mono,2,0.65,2
3,CD4 T,2,0.45,2


## Group By

`group_by` returns a grouped wrapper for one AnnData axis at a time. It can
be iterated, inspected, and used with group-local verbs.

In [23]:
grouped = adata.ap.group_by(obs="Cell_label")
grouped.group_keys()

,Cell_label
0,CD8 T
1,B cell
2,Mono
3,CD4 T


In [24]:
key, group = next(iter(grouped))
print(key)
group

{'Cell_label': 'CD8 T'}


View of AnnData object with n_obs × n_vars = 2 × 5
    obs: 'Cell_label', 'donor_id', 'sex', 'disease', 'score', 'qc_mito', 'qc_ribo'
    var: 'feature_name', 'feature_type', 'vst.mean', 'chrom'
    uns: 'title'
    obsm: 'X_pca'
    varm: 'loadings'
    layers: 'arcsinh'

In [25]:
grouped.count(sort=True)

,Cell_label,n
0,CD8 T,2
1,B cell,2
2,Mono,2
3,CD4 T,2


Grouped `mutate` evaluates window helpers such as `row_number` within each group.

In [26]:
grouped.mutate(obs={"within_label": ap.row_number(), "score_rank": ap.min_rank("score")}).obs[
    ["Cell_label", "score", "within_label", "score_rank"]
]

,Cell_label,score,within_label,score_rank
cell,,,,
cell_000,CD8 T,0.7,1.0,1.0
cell_001,B cell,0.2,1.0,1.0
cell_002,CD8 T,0.9,2.0,2.0
cell_003,Mono,0.5,1.0,1.0
cell_004,B cell,0.4,2.0,2.0
cell_005,Mono,0.8,2.0,2.0
cell_006,CD4 T,0.6,1.0,2.0
cell_007,CD4 T,0.3,2.0,1.0


In [27]:
grouped.slice_max(ap.col("score"), n=1).obs[["Cell_label", "score"]]

,Cell_label,score
cell,,
cell_002,CD8 T,0.9
cell_004,B cell,0.4
cell_005,Mono,0.8
cell_006,CD4 T,0.6


## Tidy Tables

`to_df` produces wide pandas tables for plotting or tabular inspection.
`to_tidy` produces a long table with stable observation, feature, and value
columns.

In [28]:
adata.ap.to_df(
    obs=["Cell_label", "donor_id"],
    x=["CD3E", "MS4A1"],
    obsm={"X_pca": ["0", "1"]},
).head()

,Cell_label,donor_id,CD3E,MS4A1,X_pca_0,X_pca_1
cell,,,,,,
cell_000,CD8 T,AML1,2.0,0.0,0.4,1.2
cell_001,B cell,AML1,0.0,3.0,-0.1,0.3
cell_002,CD8 T,AML2,4.0,1.0,0.8,-0.2
cell_003,Mono,AML2,1.0,4.0,1.3,0.5
cell_004,B cell,AML3,5.0,0.0,0.2,-0.8


In [29]:
tidy = adata.ap.to_tidy(obs=["Cell_label"], x=["CD3E", "MS4A1"])
tidy.head(8)

,obs_name,feature,value,Cell_label
0,cell_000,CD3E,2.0,CD8 T
1,cell_001,CD3E,0.0,B cell
2,cell_002,CD3E,4.0,CD8 T
3,cell_003,CD3E,1.0,Mono
4,cell_004,CD3E,5.0,B cell
5,cell_005,CD3E,0.0,Mono
6,cell_006,CD3E,3.0,CD4 T
7,cell_007,CD3E,2.0,CD4 T


In [30]:
ap.pivot_wider(tidy, id_cols=["obs_name", "Cell_label"], names_from="feature", values_from="value").head()

,obs_name,Cell_label,CD3E,MS4A1
0,cell_000,CD8 T,2.0,0.0
1,cell_001,B cell,0.0,3.0
2,cell_002,CD8 T,4.0,1.0
3,cell_003,Mono,1.0,4.0
4,cell_004,B cell,5.0,0.0


The pandas-only helpers cover common tidyr-style list-column and string workflows.

In [31]:
nested = ap.nest(adata.obs.reset_index(), by="Cell_label", columns=["cell", "donor_id", "score"])
nested

,Cell_label,data
0,CD8 T,cell donor_id score 0 cell_000 AM...
1,B cell,cell donor_id score 0 cell_001 AM...
2,Mono,cell donor_id score 0 cell_003 AM...
3,CD4 T,cell donor_id score 0 cell_006 AM...


In [32]:
ap.unnest(nested, "data").head()

,Cell_label,cell,donor_id,score
0,CD8 T,cell_000,AML1,0.7
1,CD8 T,cell_002,AML2,0.9
2,B cell,cell_001,AML1,0.2
3,B cell,cell_004,AML3,0.4
4,Mono,cell_003,AML2,0.5


In [33]:
sample_ids = pd.DataFrame({"sample": ["AML1_A", "AML2_B"], "markers": ["CD3E,MS4A1", "LYZ,PPBP"]})
ap.separate_rows(
    ap.separate(sample_ids, "sample", into=["donor", "tube"], sep="_"),
    "markers",
    sep=",",
)

,markers,donor,tube
0,CD3E,AML1,A
1,MS4A1,AML1,A
2,LYZ,AML2,B
3,PPBP,AML2,B


## Pipe

`pipe` passes the AnnData object into a regular Python callable, letting
wrangling chains end with project-specific steps.

In [34]:
def cell_label_counts(adata):
    """Return cell-label counts for an AnnData object."""
    return adata.ap.count("Cell_label", sort=True)


adata.ap.pipe(cell_label_counts)

,Cell_label,n
0,CD8 T,2
1,B cell,2
2,Mono,2
3,CD4 T,2


`annplyr` verbs chain naturally before the final `pipe` call.

In [35]:
(
    adata.ap.select(obs=["Cell_label", "disease"], x=["CD3E", "MS4A1"])
    .ap.filter(
        obs=ap.col("disease") == "AML",
        x=ap.if_any(ap.everything(), lambda column: ap.col(column) > 1),
    )
    .ap.pipe(lambda data: data.ap.to_df(obs=ap.everything(), x=ap.everything()))
)

,Cell_label,disease,CD3E,MS4A1
cell,,,,
cell_000,CD8 T,AML,2.0,0.0
cell_001,B cell,AML,0.0,3.0
cell_003,Mono,AML,1.0,4.0
cell_005,Mono,AML,0.0,2.0
cell_007,CD4 T,AML,2.0,5.0
